# 2 - Fine-tune DroneMegaDetector for seals

A model that finds hauled-out pinnipeds on a beach, reliably, at the
confidence you actually want to operate at.

> Environment trouble? Run `import doctor; doctor.check(fix=True)` in a
> cell, or `python doctor.py --fix` from the shell. It checks Python,
> FFmpeg, the packages, the GPU and the weights in one pass.


## Why this one should work well

Seals are **large mammals**, which is precisely what DroneMegaDetector was
built for, and hauled-out animals are a large body on open ground - the same
shape of problem as the cattle and ungulates in its training data.

So expect this fine-tune to be a modest improvement over an already-decent
baseline, not a rescue from zero. If notebook 1 showed stock DMD already
finding your seals, that is the expected result and this notebook sharpens it
for your specific coastline, light and altitude.

The gain here is mostly about **environment**, not species. MMLA's much-cited
52-point improvement came from fine-tuning across *locations*. So when you
ask an organisation for data, ask for "500 frames from your normal surveys",
not "500 pictures of seals" - easier for them, and more useful to you.

## The one design decision, stated up front

**This fine-tune does not add a class.** The output stays
`person, animal, vehicle` in DroneMegaDetector's order. A seal model
is not a model with a `seal` class; it is a model whose `animal` class
finds seal reliably.

Three reasons:

1. **Every checkpoint stays interchangeable.** One decode path, one
   `CLASS_NAMES` in `backend/lifejacket/drones/detector.py`, and switching
   models in production is three environment variables.
2. **Species naming is already somebody else's job.** The identification
   agent does it on a cropped frame, through `taxonomy.py`, which pools
   probabilities up the taxonomic tree. The detector does not need to compete
   with that.
3. **It avoids the `num_classes` trap.** Changing the head size on a
   fine-tune is a known source of silent misconfiguration in RF-DETR.

There is a cell near the end that shows what changing this would involve, if
you decide you want it.

In [ ]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dmd
import matplotlib.pyplot as plt
from dmd import load_coco, load_model, predict_images

HERE = Path.cwd().parent
DATA = Path.home() / "aerial-data"

def run(*args):
    """Call one of the sibling scripts and stream its output."""
    command = [sys.executable, *[str(a) for a in args]]
    print("$", " ".join(command[1:]), "\n")
    process = subprocess.run(command, cwd=HERE, text=True)
    if process.returncode != 0:
        raise SystemExit(f"failed: {' '.join(command[1:])}")

## Step 1 - label, then split by capture session

Export COCO from Label Studio or CVAT with the category names exactly
`person`, `animal`, `vehicle`. `dataset.py` matches on name and will stop if
anything else turns up.

**Splits are by capture session, never random.** Consecutive frames of one
animal are near-identical; split them randomly and the same animal lands in
train and test, so the model has effectively seen the answers and the score
means nothing.

Run `--inspect` first and read the session list.

In [ ]:
RAW = DATA / "seals_labelled"
SPLIT = DATA / "seals"

run(HERE / "dataset.py", "--coco", RAW / "_annotations.coco.json",
    "--images", RAW, "--out", SPLIT, "--inspect")

If the session keys above look like individual frames rather than capture
sessions, stop: your test split will leak. Fix it with `--group-regex`
before writing anything.

If a split came out empty, you do not have enough sessions. The script will
refuse to write rather than hand you a dataset that cannot train.

In [ ]:
run(HERE / "dataset.py", "--coco", RAW / "_annotations.coco.json",
    "--images", RAW, "--out", SPLIT, "--overwrite")

## Step 2 - look at the data before training on it

Cheap, and it catches the mistakes that are expensive later: boxes on the
wrong objects, a whole session mislabelled, targets far too small.

In [ ]:
train_set = load_coco(SPLIT / "train" / "_annotations.coco.json")
print(f"train: {len(train_set)} images, {train_set.box_count} boxes")
print("per class:", train_set.per_class_counts())

sizes = dmd.pixels_on_target([b for boxes in train_set.boxes.values() for b in boxes])
figure, axis = plt.subplots(figsize=(7, 3.5))
axis.hist(sizes, bins=30, color="#81C784", edgecolor="#555")
axis.axvline(32, color="#C62828", linestyle="--", label="~32 px detector floor")
axis.set_xlabel("longest side of a box (px)")
axis.legend()
plt.show()

sample = sorted(train_set.images)[:6]
dmd.show_grid(
    [dmd.draw_boxes(train_set.image_path(i), None, train_set.boxes.get(i, []))
     for i in sample],
    columns=3,
    titles=[train_set.image_path(i).name for i in sample],
)
plt.show()

## Step 3 - where is the model now, before training?

You cannot claim an improvement without this number.

In [ ]:
TEST = SPLIT / "test" / "_annotations.coco.json"

run(HERE / "evaluate.py", "--weights", "dmd", "--data", TEST, "--tag", "seals-v1-before")

## Step 4 - fine-tune, with replay

**Replay matters.** Fine-tuning only on seal makes the model better at
seal and measurably worse at everything else, including `person` and
`vehicle`, which every site needs. That is catastrophic forgetting.

`--replay DIR` mixes a slice of general data into the training set to hold it
off. Point it at any dataset in the same layout - your NOAA set, or a general
drone set. The validation split stays purely task-specific, because what you
want to know is whether the model got better at *this* job.

If you have no general set to hand, lower the learning rate to `5e-5` and
shorten training instead. Less adaptation, but less forgetting too.

In [ ]:
REPLAY = DATA / "general"     # set to None if you have nothing to mix in
EPOCHS = 40
LR = 0.0001

arguments = [HERE / "finetune.py", "--data", SPLIT, "--name", "seals-v1",
             "--epochs", EPOCHS, "--lr", LR]
if REPLAY and REPLAY.exists():
    arguments += ["--replay", REPLAY, "--replay-fraction", 0.25]
else:
    print("no replay set found - training on the specific data alone.")
    print("Watch the general-regression check in step 6 closely.\n")

run(*arguments, "--dry-run")      # check the layout first; drop --dry-run to train

The dry run checks the dataset layout, the class order and the base weights
without starting training. If it passed, train for real.

**On a GPU VM, run the real training in `tmux` from a terminal, not here.**
A notebook cell holding a multi-hour run will die when the browser proxy
drops, and Workbench's idle shutdown counts kernel activity, not shell
activity. The command is printed below - copy it.

In [ ]:
print("tmux new -s seals-v1")
print(f"cd {HERE}")
print("python finetune.py \\")
print(f"    --data {SPLIT} \\")
print(f"    --name seals-v1 --epochs {EPOCHS} --lr {LR} \\")
print("    --replay ~/aerial-data/general --replay-fraction 0.25")
print("\n# Ctrl-b d to detach, tmux attach -t seals-v1 to come back")

In [ ]:
# Or run it here anyway, for a short job. Uncomment to use.
# run(*arguments)

## Step 5 - did it get better at the job?

In [ ]:
RUN = HERE / "runs" / "seals-v1"
candidates = sorted(RUN.glob("checkpoint_best*.pth")) or sorted(RUN.glob("*.pth"))
BEST = candidates[0] if candidates else None
print("best weights:", BEST)

if BEST:
    run(HERE / "evaluate.py", "--weights", BEST, "--data", TEST,
        "--tag", "seals-v1-after",
        "--compare", HERE / "results" / "eval_seals-v1-before.json")

## Step 6 - did it get worse at everything else?

**Do not skip this.** A seal model that stopped detecting people is a
worse product in the field than stock DroneMegaDetector, and without this
check you would ship it not knowing.

`evaluate.py --compare` flags any metric that fell by 0.05 or more.

In [ ]:
GENERAL = DATA / "noaa_eval" / "_annotations.coco.json"

if BEST and GENERAL.exists():
    run(HERE / "evaluate.py", "--weights", BEST, "--data", GENERAL,
        "--tag", "seals-v1-general",
        "--compare", HERE / "results" / "eval_noaa-dmd.json")
else:
    print("No general evaluation set. Build one with notebook 1 before publishing.")

If that reported regressions: raise `--replay-fraction` to 0.4, or drop the
learning rate to `5e-5`, and train again. Both trade a little task accuracy
for retained general ability, which is almost always the right trade for a
dispatch tool.

## Step 7 - look at before and after, side by side

Numbers tell you whether; pictures tell you why.

In [ ]:
if BEST:
    test_set = load_coco(TEST)
    ids = sorted(test_set.images)[:4]
    paths = [test_set.image_path(i) for i in ids]

    before = predict_images(load_model(), paths, threshold=0.05)
    after = predict_images(load_model(BEST), paths, threshold=0.05)

    THRESHOLD = 0.3
    panels, titles = [], []
    for i, path in zip(ids, paths, strict=True):
        gt = test_set.boxes.get(i, [])
        panels.append(dmd.draw_boxes(path, before[path], gt, threshold=THRESHOLD))
        titles.append(f"{path.name} - stock DMD")
        panels.append(dmd.draw_boxes(path, after[path], gt, threshold=THRESHOLD))
        titles.append(f"{path.name} - seals-v1")
    dmd.show_grid(panels, columns=2, titles=titles)
    plt.show()

## If you want a dedicated `seal` class after all

You probably do not - see the decision at the top. But if the scene summary
genuinely needs to distinguish "a seal" from "an animal", here is the whole
change:

1. `class_map.yaml` - add `seal` to `targets`
2. `dmd.py` - `DMD_CLASSES = ("person", "animal", "vehicle", "seal")`
3. `backend/lifejacket/drones/detector.py` - the same tuple
4. `publish.py` - the same tuple
5. `backend/tests/test_drone_detector.py` - the decode tests build tensors
   with `4 + len(CLASS_NAMES)` channels and will need updating

And then in `finetune.py`, pass `num_classes=4` to the `RFDETRMedium`
constructor - RF-DETR will not infer it reliably on its own
([issue #878](https://github.com/roboflow/rf-detr/issues/878)).

The real cost is not the code. It is that you can no longer swap this model
for any other, and the app needs per-model class metadata.

## What good looks like here

- **Recall up, precision roughly held.** If precision collapsed, the model is
  now calling sand and foam "animal". Check the false positives by eye.
- **The best-F1 threshold moved down.** A more confident model lets you
  operate at a lower threshold, which means fewer misses.
- **`blind_images` near zero.** For a stranding tool, a frame with an animal
  and no detection is the failure that matters.

## Step 8 - publish, if it earned it

Only publish a checkpoint that improved on its task **and** did not regress
on the general set.

```bash
python publish.py --weights runs/seals-v1/checkpoint_best_total.pth \
    --repo YOUR_USERNAME/lifejacket-seal --private
```

That prints a `DRONE_DETECTOR_*` block for `.env`. The app downloads by
pinned commit SHA and verifies the hash on every load.